[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/06_multihead_attention.ipynb)

# 🔴 Hard: Multi-Head Attention

Implement **Multi-Head Attention** from scratch — the core building block of the Transformer.

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \dots, \text{head}_h) W^O$$
$$\text{head}_i = \text{Attention}(Q W_i^Q,\; K W_i^K,\; V W_i^V)$$

### Signature
```python
class MultiHeadAttention:
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, Q, K, V) -> torch.Tensor: ...
```

### Requirements
- Use `nn.Linear(d_model, d_model)` for `self.W_q`, `self.W_k`, `self.W_v`, `self.W_o`
- `d_k = d_model // num_heads` per head
- `forward(Q, K, V)`: Q is `(B, seq_q, d_model)`, K/V are `(B, seq_k, d_model)`
- Must support **cross-attention** (`seq_q != seq_k`)
- Do **NOT** use `torch.nn.MultiheadAttention`
- You **may** use `torch.softmax` and `torch.matmul`

### Steps
1. Project: `q = self.W_q(Q)`, `k = self.W_k(K)`, `v = self.W_v(V)`
2. Reshape to `(B, num_heads, seq, d_k)`
3. Scaled dot-product attention per head
4. Concat heads → `(B, seq_q, d_model)`
5. Output projection: `self.W_o(concat)`

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.0 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import math

In [20]:
# ✏️ YOUR IMPLEMENTATION HERE

class MultiHeadAttention:
    def __init__(self, d_model: int, num_heads: int):
        # First make sure the hid dim is divisable by the number of heads to get the head dim per head (Splits evenly across heads)
        assert(d_model % num_heads == 0)

        self.num_heads = num_heads
        self.d_model = d_model
        self.head_dim = d_model // num_heads

        # The projections (Creating the Q, k, V from input features)
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)

        # The final output projection
        self.W_o = nn.Linear(d_model, d_model) # A Linear layer to combine the heads outputs

    def forward(self, Q, K, V):
        b, seq_q, d_model = Q.shape

        # Project inputs
        Q = self.W_q(Q) # [b, seq_q, d_model]
        K = self.W_k(K) # [b, seq_k, d_model]
        V = self.W_v(V) # [b, seq_k, d_model]

        # Enroll the last dim >> num_heads x head_dim >> then we exchange the places of the seq length with the num heads so that the num_heads to be part of the batch size, and we do our calculations across all heads
        Q = Q.view(b, -1, self.num_heads, self.head_dim).transpose(1, 2) # [b, seq_q, d_model] >> [b, seq_q, num_heads, head_dim] >> [b, num_heads, seq_q, head_dim]
        K = K.view(b, -1, self.num_heads, self.head_dim).transpose(1, 2) # [b, seq_k, d_model] >> [b, seq_k, num_heads, head_dim] >> [b, num_heads, seq_k, head_dim]
        V = V.view(b, -1, self.num_heads, self.head_dim).transpose(1, 2) # [b, seq_k, d_model] >> [b, seq_k, num_heads, head_dim] >> [b, num_heads, seq_k, head_dim]


        # Get the last dim of k
        # d_k = K.shape[-1]
        # Attention scores
        atten_scores = Q @ K.transpose(2, 3) # [b, num_heads, seq_q, head_dim] @ [b, num_heads, head_dim, seq_k] >> [b, num_heads, seq_q, seq_k]
        # We scale to avoid vanishing gradient and for more training stability
        atten_scores = atten_scores / math.sqrt(self.head_dim)

        # Attention weights >> we do that across the final dim, So that each query attend to all keys and the percentage sumation equals to 1 (100$)
        atten_weights = torch.softmax(atten_scores, dim=-1) # [b, num_heads, seq_q, seq_k]

        # Get the cotext vector
        context_vec = atten_weights @ V # [b, num_heads, seq_q, seq_k] @ [b, num_heads, seq_k, head_dim] >> [b, num_heads, seq_q, head_dim]
        context_vec = context_vec.transpose(1, 2) # [b, seq_q, num_heads, head_dim]

        # Convert it back to 3 dim
        context_vec = context_vec.reshape(b, -1, self.d_model) # [b, seq_q, d_model]

        # Final Linear projection
        output = self.W_o(context_vec)

        return output



In [21]:
# 🧪 Debug
torch.manual_seed(0)
mha = MultiHeadAttention(d_model=32, num_heads=4)
print("W_q type:", type(mha.W_q))          # should be nn.Linear
print("W_q.weight shape:", mha.W_q.weight.shape)  # (32, 32)

x = torch.randn(2, 6, 32)
out = mha.forward(x, x, x)
print("Output shape:", out.shape)          # (2, 6, 32)

# Cross-attention
Q = torch.randn(1, 3, 32)
K = torch.randn(1, 7, 32)
V = torch.randn(1, 7, 32)
out2 = mha.forward(Q, K, V)
print("Cross-attn shape:", out2.shape)     # (1, 3, 32)

W_q type: <class 'torch.nn.modules.linear.Linear'>
W_q.weight shape: torch.Size([32, 32])
Output shape: torch.Size([2, 6, 32])
Cross-attn shape: torch.Size([1, 3, 32])


In [22]:
# ✅ SUBMIT
from torch_judge import check
check("mha")


🧪 Testing: Multi-Head Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/6] Output shape (7.3ms)
  ✅ [2/6] Uses nn.Linear with correct shapes (1.4ms)
  ✅ [3/6] Numerical correctness vs reference (4.8ms)
  ✅ [4/6] Gradient flow (5.3ms)
  ✅ [5/6] Cross-attention (seq_q != seq_k) (1.4ms)
  ✅ [6/6] Different heads give different outputs (4.5ms)
──────────────────────────────────────────────────
  🎉 All 6 tests passed! (24.7ms total)
  Progress saved. Run status() to see your dashboard.

